In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ y = \text{onehot eg.: [1, 0, 0] or [0, 1, 0] or [0, 0, 1]}  $$
$$ y_k \in \{0, 1\}, \quad \sum_k y_k = 1 $$
$$ \\[2em] $$
$$ Model = M(W, b) = Wx + b $$
$$ \frac{\partial M}{\partial W} = x $$
$$ \frac{\partial M}{\partial b} = 1 $$
$$ \\[2em] $$
$$ Softmax(M_i) = S_i = \frac{e^{M_i}}{\sum_{k} e^{M_k}} $$
$$
\frac{\partial S_i}{\partial M_j} = S_i (\delta_{ij} - S_j) =
\begin{dcases}
i = j & \frac{\partial S_i}{\partial M_i} = 
\frac{e^{M_i} \sum_{k} e^{M_k} - e^{M_i} e^{M_i}}{\sum_{k} e^{M_k} \sum_{k} e^{M_k}} =
S_i (1 - S_i) \\
\\
i \neq j & \frac{\partial S_i}{\partial M_j} = 
\frac{0 \cdot \sum_{k} e^{M_k} - e^{M_i} e^{M_j}}{\sum_{k} e^{M_k} \sum_{k} e^{M_k}} = 
-S_i S_j  = S_i (0 - S_j)\\
\end{dcases}
$$
$$ \\[2em] $$
$$ Loss = L = -\sum_{i} y_i \log S_i $$
$$ \frac{\partial L}{\partial S_i} = - y_i \frac{1}{S_i} $$
$$ \frac{\partial L}{\partial S} = \Bigg[ - y_0 \frac{1}{S_0}, - y_1 \frac{1}{S_1}, \dots, - y_n \frac{1}{S_n} \Bigg] = -y_{onehot} \cdot \frac{1}{S} $$
$$ \\[1em] $$
$$ \frac{\partial L}{\partial M_j} = \sum_{i} \frac{\partial L}{\partial S_i} \frac{\partial S_i}{\partial M_j} $$
$$ \frac{\partial L}{\partial M_j} = 
\sum_{i} \Big(- y_i \frac{1}{S_i}\Big) \cdot S_i (\delta_{ij} - S_j) = 
-\sum_{i} y_i \delta_{ij} + \sum_{i} y_i S_j = S_j - y_j $$
$$ \frac{\partial L}{\partial M} = \Bigg[ S_0 - y_0, S_1 - y_1, \dots, S_n - y_n \Bigg] = S - y_{onehot} $$
$$ \frac{\partial L}{\partial W} = \frac{\partial L}{\partial M} \cdot \frac{\partial M}{\partial W} = (S - y_{onehot}) \cdot x^T $$
$$ \frac{\partial L}{\partial b} = \frac{\partial L}{\partial M} \cdot \frac{\partial M}{\partial b} = S - y_{onehot} $$

In [ ]:
from torch import arange, exp, log, randint, randn, zeros, Tensor

import import_ipynb
from common import assert_eq, Patient, T # type: ignore


def _Model(X: Tensor, w: Tensor, b: Tensor) -> Tensor:
    return X @ w.T + b

t = Tensor([[1.0, 2.0, 3.0, 4.0, 5.0, 6.0], [3.0, 4.0, 5.0, 6.0, 7.0, 8.0]])

t_view = t[:, [0, 1, 2, 3, 5]]


def _Softmax(M: Tensor) -> Tensor:
    z_shift = M - M.max(dim=1, keepdim=True).values
    exp_z = exp(z_shift)
    return exp_z / exp_z.sum(dim=1, keepdim=True)

def logistic_regression_nc_sgd_gradient(X: Tensor, y: Tensor, epochs=2000, lr=0.1) -> tuple[float, callable]:
    (s, f) = X.shape
    
    w = randn(K, D, requires_grad=False) * 0.01
    b = zeros(K, requires_grad=False)

    for _ in range(200):
        M = _Model(X, w, b)
        p = _Softmax(M)

        grad_z = p.clone()
        grad_z[arange(s), y] -= 1
        grad_z /= s

        grad_W = grad_z.T @ X          
        grad_b = grad_z.sum(dim=0)    

        w -= lr * grad_W
        b -= lr * grad_b
        
        loss = -log(p[arange(s), y]).mean()
